# Lab 03: Data Cleaning and Transformation Pipeline
**Name:** Claudio Bayro


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg, count, sum
from spark_utils import SparkUtils


In [ ]:
spark_utils = SparkUtils()
spark = spark_utils.get_spark_session("Lab03-QuickCommerce")


In [ ]:
df = spark.read.csv("data/quick_commerce.csv", header=True, inferSchema=True)
df.printSchema()
df.show(5)


## Dataset Cleaning


In [ ]:
null_counts = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])
null_counts.show()


In [ ]:
df = df.fillna({
    "Delivery_Time_Min": 0,
    "Discount_Applied": 0,
    "Customer_Age": 0,
    "Order_Value": 0
})


## Task 1: Delivery SLA Classification


In [ ]:
df_sla = df.withColumn(
    "Delivery_SLA",
    when(col("Delivery_Time_Min") <= 15, "FAST")
    .when((col("Delivery_Time_Min") > 15) & (col("Delivery_Time_Min") <= 20), "ON_TIME")
    .otherwise("LATE")
)


In [ ]:
late_orders = df_sla.filter(col("Delivery_SLA") == "LATE") \
    .orderBy(col("Delivery_Time_Min").desc())

late_orders.select(
    "Order_ID",
    "Company",
    "City",
    "Delivery_Time_Min",
    "Delivery_SLA"
).show()


## Task 2: Discount Impact


In [ ]:
df_price = df_sla.withColumn(
    "Effective_Order_Value",
    col("Order_Value") * (1 - col("Discount_Applied"))
)


In [ ]:
df_price = df_price.withColumn(
    "Price_Bucket",
    when(col("Effective_Order_Value") < 200, "LOW")
    .when((col("Effective_Order_Value") >= 200) & (col("Effective_Order_Value") <= 500), "MEDIUM")
    .otherwise("HIGH")
)


In [ ]:
price_analysis = df_price.groupBy("Price_Bucket") \
    .agg(
        count("*").alias("total_orders"),
        avg("Effective_Order_Value").alias("avg_effective_value")
    ) \
    .orderBy(col("avg_effective_value").desc())

price_analysis.show()


## Task 3: Customer Segmentation


In [ ]:
df_valid_age = df_price.filter(
    (col("Customer_Age").isNotNull()) &
    (col("Customer_Age") >= 0) &
    (col("Customer_Age") <= 100)
)


In [ ]:
df_segmented = df_valid_age.withColumn(
    "Age_Group",
    when(col("Customer_Age") < 25, "YOUNG")
    .when((col("Customer_Age") >= 25) & (col("Customer_Age") <= 44), "ADULT")
    .otherwise("SENIOR")
)


In [ ]:
segment_analysis = df_segmented.groupBy("Age_Group", "Product_Category") \
    .agg(
        count("*").alias("orders"),
        avg("Order_Value").alias("avg_order_value")
    ) \
    .orderBy(col("Age_Group").asc(), col("orders").desc())

segment_analysis.show()
